# 🎯 LLM Classification Fine-Tuning - Comprehensive EDA & Problem Framing

## 📋 Competition Objective & Problem Framing

### **🎪 What is this Competition About?**

This is a **Human Preference Prediction** competition where we need to predict which AI-generated response humans would prefer when given two options.

**The Core Challenge:**
- **Input**: A prompt (question/instruction) + Two AI responses (Response A & Response B)
- **Output**: Probability distribution across 3 outcomes:
  - `winner_model_a` - Probability that Response A is preferred
  - `winner_model_b` - Probability that Response B is preferred  
  - `winner_tie` - Probability that both responses are equally good
- **Constraint**: All probabilities must sum to 1.0

### **🎯 Why This Matters**
1. **AI Safety & Alignment**: Understanding human preferences is crucial for building helpful AI
2. **Model Evaluation**: Automated ways to assess AI response quality
3. **Business Applications**: Chatbot improvement, content ranking, recommendation systems

### **🧠 Key Insights from Similar Competitions**
- **LMSYS Chatbot Arena**: Similar format, winners used knowledge distillation + external data
- **Human preferences are subjective** but have patterns (length, formatting, correctness)
- **Comparative analysis** is more important than absolute quality assessment
- **Advanced techniques** like LoRA fine-tuning and pseudo-labeling are game-changers

---

## 📊 Analysis Approach

This notebook will systematically analyze:
1. **Data Structure & Quality** - Understanding the format and completeness
2. **Text Characteristics** - Length, complexity, content patterns
3. **Target Distribution** - Preference patterns and confidence levels
4. **Feature Engineering** - Creating predictive features for modeling
5. **Advanced Analysis** - Readability, sentiment, and quality indicators
6. **Modeling Recommendations** - Strategic approach based on findings


In [ ]:
!pip install textstat

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
import warnings
from collections import Counter, defaultdict
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string

warnings.filterwarnings('ignore')

# Try to import optional libraries
try:
    from textstat import flesch_reading_ease, flesch_kincaid_grade, automated_readability_index
    TEXTSTAT_AVAILABLE = True
    print("✅ textstat available for readability analysis")
except ImportError:
    print("⚠️  textstat not available. Install with: pip install textstat")
    TEXTSTAT_AVAILABLE = False

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
    print("✅ wordcloud available for text visualization")
except ImportError:
    print("⚠️  wordcloud not available. Install with: pip install wordcloud")
    WORDCLOUD_AVAILABLE = False

# Set visualization style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = [15, 8]
plt.rcParams['font.size'] = 10

# 1. 📁 Data Loading & Initial Exploration

In [ ]:
# Load all datasets
print("📂 Loading competition datasets...")

try:
    train_df = pd.read_csv('/kaggle/input/llm-classification-finetuning/train.csv')
    test_df = pd.read_csv('/kaggle/input/llm-classification-finetuning/test.csv')
    sample_submission = pd.read_csv('/kaggle/input/llm-classification-finetuning/sample_submission.csv')
    
    print(f"✅ Training set: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
    print(f"✅ Test set: {test_df.shape[0]:,} rows × {test_df.shape[1]} columns")
    print(f"✅ Sample submission: {sample_submission.shape[0]:,} rows × {sample_submission.shape[1]} columns")
    
    # Memory usage analysis
    train_memory = train_df.memory_usage(deep=True).sum() / 1024**2
    print(f"📊 Training data memory usage: {train_memory:.1f} MB")
    
except FileNotFoundError as e:
    print(f"❌ Error loading files: {e}")
    print("📁 Make sure you're running this notebook from the eda/ directory")
    print("📁 And that train.csv, test.csv, sample_submission.csv are in the parent directory")

print("\n" + "="*50)
print("📋 DATASET OVERVIEW")
print("="*50)

In [ ]:
# Detailed data structure analysis
print("🔍 TRAINING DATA STRUCTURE")
print("-" * 30)
print(f"Shape: {train_df.shape}")
print(f"Columns: {list(train_df.columns)}")
print("\nData Types:")
print(train_df.dtypes)

print("\n🔍 MISSING VALUES ANALYSIS")
print("-" * 30)
missing_train = train_df.isnull().sum()
missing_test = test_df.isnull().sum()

print("Training set missing values:")
for col, count in missing_train.items():
    if count > 0:
        print(f"  {col}: {count} ({count/len(train_df)*100:.2f}%)")
    else:
        print(f"  {col}: ✅ No missing values")

if len(test_df) > 0:
    print("\nTest set missing values:")
    for col, count in missing_test.items():
        if count > 0:
            print(f"  {col}: {count} ({count/len(test_df)*100:.2f}%)")
        else:
            print(f"  {col}: ✅ No missing values")

# Check for target columns
target_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
has_targets = all(col in train_df.columns for col in target_cols)

print(f"\n🎯 TARGET COLUMNS PRESENT: {'✅ Yes' if has_targets else '❌ No'}")

if has_targets:
    print("\nTarget Statistics:")
    for col in target_cols:
        mean_val = train_df[col].mean()
        std_val = train_df[col].std()
        min_val = train_df[col].min()
        max_val = train_df[col].max()
        print(f"  {col}: μ={mean_val:.4f}, σ={std_val:.4f}, range=[{min_val:.4f}, {max_val:.4f}]")
        
    # Check if probabilities sum to 1
    prob_sums = train_df[target_cols].sum(axis=1)
    print(f"\n📊 Probability sums: μ={prob_sums.mean():.6f}, σ={prob_sums.std():.6f}")
    print(f"✅ All probabilities sum to ~1.0: {np.allclose(prob_sums, 1.0, atol=1e-6)}")
else:
    print("ℹ️  This appears to be a test-only or unlabeled dataset format")

In [ ]:
# Sample data exploration
print("📋 SAMPLE DATA EXAMINATION")
print("-" * 30)

if len(train_df) > 0:
    print("First training example:")
    print(f"ID: {train_df.iloc[0]['id']}")
    
    # Show raw data format
    for col in ['prompt', 'response_a', 'response_b']:
        if col in train_df.columns:
            value = str(train_df.iloc[0][col])
            print(f"\n{col.upper()} (raw):")
            print(f"  Type: {type(train_df.iloc[0][col])}")
            print(f"  Length: {len(value)} characters")
            print(f"  Preview: {value[:200]}...")
    
    if has_targets:
        print(f"\nTARGETS:")
        for col in target_cols:
            print(f"  {col}: {train_df.iloc[0][col]:.4f}")
        
        # Determine winner
        max_idx = train_df.iloc[0][target_cols].idxmax()
        max_val = train_df.iloc[0][target_cols].max()
        print(f"  🏆 Winner: {max_idx} (confidence: {max_val:.4f})")

print("\n📊 SAMPLE SUBMISSION FORMAT")
print("-" * 30)
print(sample_submission.head())
print(f"\nBaseline predictions: All equal probabilities ({1/3:.4f} each)")

# Basic statistics
print(f"\n📈 BASIC STATISTICS")
print("-" * 30)
print(f"Unique IDs in train: {train_df['id'].nunique():,}")
print(f"Unique IDs in test: {test_df['id'].nunique():,}")
print(f"ID overlap between train/test: {len(set(train_df['id']) & set(test_df['id']))}")

if len(train_df) > 0:
    duplicated_ids = train_df['id'].duplicated().sum()
    print(f"Duplicated IDs in train: {duplicated_ids}")
    if duplicated_ids > 0:
        print("⚠️  Warning: Found duplicate IDs in training data!")

display(train_df.head())

In [ ]:
# Sample data exploration
print("📋 SAMPLE DATA EXAMINATION")
print("-" * 30)

if len(train_df) > 0:
    print("First training example:")
    print(f"ID: {train_df.iloc[0]['id']}")
    
    # Show raw data format
    for col in ['prompt', 'response_a', 'response_b']:
        if col in train_df.columns:
            value = str(train_df.iloc[0][col])
            print(f"\n{col.upper()} (raw):")
            print(f"  Type: {type(train_df.iloc[0][col])}")
            print(f"  Length: {len(value)} characters")
            print(f"  Preview: {value[:200]}...")
    
    if has_targets:
        print(f"\nTARGETS:")
        for col in target_cols:
            print(f"  {col}: {train_df.iloc[0][col]:.4f}")
        
        # Determine winner
        max_idx = train_df.iloc[0][target_cols].idxmax()
        max_val = train_df.iloc[0][target_cols].max()
        print(f"  🏆 Winner: {max_idx} (confidence: {max_val:.4f})")

print("\n📊 SAMPLE SUBMISSION FORMAT")
print("-" * 30)
print(sample_submission.head())
print(f"\nBaseline predictions: All equal probabilities ({1/3:.4f} each)")

# Basic statistics
print(f"\n📈 BASIC STATISTICS")
print("-" * 30)
print(f"Unique IDs in train: {train_df['id'].nunique():,}")
print(f"Unique IDs in test: {test_df['id'].nunique():,}")
print(f"ID overlap between train/test: {len(set(train_df['id']) & set(test_df['id']))}")

if len(train_df) > 0:
    duplicated_ids = train_df['id'].duplicated().sum()
    print(f"Duplicated IDs in train: {duplicated_ids}")
    if duplicated_ids > 0:
        print("⚠️  Warning: Found duplicate IDs in training data!")

display(train_df.head())

# 2. 🔄 Text Processing & Feature Extraction

In [ ]:
# Comprehensive text processing functions
def safe_json_loads(json_str):
    """Safely parse JSON strings with error handling"""
    try:
        if pd.isna(json_str):
            return None
        return json.loads(json_str)
    except (json.JSONDecodeError, TypeError):
        return json_str  # Return original if not valid JSON

def extract_text_content(parsed_content):
    """Extract text content from parsed JSON or return as-is"""
    if parsed_content is None:
        return ""
    if isinstance(parsed_content, list) and len(parsed_content) > 0:
        return str(parsed_content[0])
    return str(parsed_content)

def calculate_text_features(text):
    """Calculate comprehensive text features"""
    if pd.isna(text) or text == "":
        return {
            'char_count': 0, 'word_count': 0, 'sentence_count': 0, 'paragraph_count': 0,
            'avg_word_length': 0, 'punct_count': 0, 'uppercase_count': 0, 'digit_count': 0
        }
    
    text = str(text)
    
    # Basic counts
    char_count = len(text)
    words = text.split()
    word_count = len(words)
    sentence_count = len(re.findall(r'[.!?]+', text)) + 1
    paragraph_count = text.count('\n\n') + 1
    
    # Advanced features
    avg_word_length = np.mean([len(word) for word in words]) if words else 0
    punct_count = sum(1 for char in text if char in string.punctuation)
    uppercase_count = sum(1 for char in text if char.isupper())
    digit_count = sum(1 for char in text if char.isdigit())
    
    return {
        'char_count': char_count,
        'word_count': word_count,
        'sentence_count': sentence_count,
        'paragraph_count': paragraph_count,
        'avg_word_length': avg_word_length,
        'punct_count': punct_count,
        'uppercase_count': uppercase_count,
        'digit_count': digit_count
    }

def detect_content_types(text):
    """Detect various content types in text"""
    if pd.isna(text) or text == "":
        return {
            'has_code': False, 'has_markdown': False, 'has_links': False, 
            'has_lists': False, 'has_tables': False, 'has_math': False
        }
    
    text = str(text)
    
    # Code detection patterns
    code_patterns = [
        r'```[\s\S]*?```',  # Code blocks
        r'`[^`]+`',         # Inline code
        r'\bdef\s+\w+\s*\(',  # Python functions
        r'\bclass\s+\w+\s*[:\(]',  # Python classes
        r'\bimport\s+\w+',  # Import statements
        r'\bfunction\s+\w+\s*\(',  # JavaScript functions
        r'<[^>]+>',         # HTML/XML tags
    ]
    
    has_code = any(re.search(pattern, text, re.IGNORECASE) for pattern in code_patterns)
    
    # Markdown patterns
    markdown_patterns = [
        r'\*\*[^*]+\*\*',   # Bold
        r'\*[^*]+\*',       # Italic
        r'^#{1,6}\s',       # Headers
        r'^\s*[-*+]\s',     # Lists
    ]
    
    has_markdown = any(re.search(pattern, text, re.MULTILINE) for pattern in markdown_patterns)
    
    # Other content types
    has_links = bool(re.search(r'https?://\S+', text))
    has_lists = bool(re.search(r'^\s*(\d+\.|\*|-|\+)\s', text, re.MULTILINE))
    has_tables = bool(re.search(r'\|.*\|', text))
    has_math = bool(re.search(r'\$[^$]+\$|\\\(.*?\\\)|\\\[.*?\\\]', text))
    
    return {
        'has_code': has_code,
        'has_markdown': has_markdown,
        'has_links': has_links,
        'has_lists': has_lists,
        'has_tables': has_tables,
        'has_math': has_math
    }

def calculate_readability_scores(text):
    """Calculate readability scores if textstat is available"""
    if not TEXTSTAT_AVAILABLE or pd.isna(text) or text == "":
        return {'flesch_ease': np.nan, 'flesch_kincaid': np.nan, 'automated_readability': np.nan}
    
    try:
        text = str(text)
        if len(text.strip()) == 0:
            return {'flesch_ease': np.nan, 'flesch_kincaid': np.nan, 'automated_readability': np.nan}
        
        return {
            'flesch_ease': flesch_reading_ease(text),
            'flesch_kincaid': flesch_kincaid_grade(text),
            'automated_readability': automated_readability_index(text)
        }
    except:
        return {'flesch_ease': np.nan, 'flesch_kincaid': np.nan, 'automated_readability': np.nan}


In [ ]:
# Process and extract text features
print("⚡ Processing text data and extracting features...")

# Parse JSON columns
for dataset_name, df in [("Training", train_df), ("Test", test_df)]:
    print(f"\n📊 Processing {dataset_name} dataset...")
    
    # Parse JSON fields
    for col in ['prompt', 'response_a', 'response_b']:
        if col in df.columns:
            print(f"  Parsing {col}...")
            df[f'parsed_{col}'] = df[col].apply(safe_json_loads)
            df[f'{col}_text'] = df[f'parsed_{col}'].apply(extract_text_content)
    
    # Extract text features for each text field
    text_cols = ['prompt_text', 'response_a_text', 'response_b_text']
    
    for text_col in text_cols:
        if text_col in df.columns:
            print(f"  Extracting features from {text_col}...")
            
            # Basic text features
            features = df[text_col].apply(calculate_text_features)
            feature_df = pd.DataFrame(features.tolist())
            for feature_name in feature_df.columns:
                df[f'{text_col}_{feature_name}'] = feature_df[feature_name]
            
            # Content type detection
            content_types = df[text_col].apply(detect_content_types)
            content_df = pd.DataFrame(content_types.tolist())
            for content_name in content_df.columns:
                df[f'{text_col}_{content_name}'] = content_df[content_name]
            
            # Readability scores (sample for performance)
            if TEXTSTAT_AVAILABLE and len(df) > 0:
                sample_size = min(1000, len(df))
                sample_indices = np.random.choice(len(df), sample_size, replace=False)
                
                readability_scores = df.iloc[sample_indices][text_col].apply(calculate_readability_scores)
                readability_df = pd.DataFrame(readability_scores.tolist(), index=sample_indices)
                
                for score_name in readability_df.columns:
                    df[f'{text_col}_{score_name}'] = np.nan
                    df.loc[sample_indices, f'{text_col}_{score_name}'] = readability_df[score_name]

print("\n✅ Text processing completed!")

# ========================================================
# 📊 CREATE COMPREHENSIVE FEATURES DATAFRAME - FIXED
# ========================================================
def safe_round(value, decimals=3):
    """Safely round values, handling numpy types"""
    try:
        if pd.isna(value):
            return 'N/A'
        return round(float(value), decimals)
    except (TypeError, ValueError):
        return 'N/A'

def safe_convert(value, max_length=50):
    """Safely convert values to string with length limit"""
    try:
        if pd.isna(value):
            return 'N/A'
        return str(value)[:max_length]
    except:
        return 'N/A'

if len(train_df) > 0:
    print("\n📋 COMPREHENSIVE FEATURES ANALYSIS")
    print("=" * 60)
    
    # Get all feature columns (excluding original and target columns)
    original_cols = ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b']
    target_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
    parsed_cols = [col for col in train_df.columns if col.startswith('parsed_')]
    
    # Extract all feature columns
    feature_cols = [col for col in train_df.columns 
                    if col not in original_cols + target_cols + parsed_cols]
    
    # Create comprehensive features summary dataframe
    features_summary = []
    for col in feature_cols:
        col_data = train_df[col]
        
        # Determine feature category
        if col.startswith('prompt_text_'):
            category = 'Prompt Features'
        elif col.startswith('response_a_text_'):
            category = 'Response A Features'
        elif col.startswith('response_b_text_'):
            category = 'Response B Features'
        elif any(x in col for x in ['char_count', 'word_count', 'sent_count']):
            category = 'Length Features'
        elif any(x in col for x in ['has_', 'contains_']):
            category = 'Content Detection'
        elif any(x in col for x in ['flesch', 'readability', 'grade']):
            category = 'Readability Features'
        else:
            category = 'Other Features'
        
        # Calculate statistics safely
        non_null_count = int(col_data.count())
        missing_count = int(col_data.isnull().sum())
        missing_percentage = safe_round((missing_count / len(col_data)) * 100, 2)
        unique_values = int(col_data.nunique())
        
        # Handle numeric vs non-numeric columns
        if col_data.dtype in ['int64', 'float64']:
            min_val = safe_round(col_data.min(), 3)
            max_val = safe_round(col_data.max(), 3)
            mean_val = safe_round(col_data.mean(), 3)
            std_val = safe_round(col_data.std(), 3)
        else:
            min_val = safe_convert(col_data.min(), 30)
            max_val = safe_convert(col_data.max(), 30)
            mean_val = 'N/A'
            std_val = 'N/A'
        
        # Get sample value
        sample_val = safe_convert(col_data.dropna().iloc[0] if not col_data.dropna().empty else 'N/A', 50)
        
        features_summary.append({
            'Feature_Name': col,
            'Category': category,
            'Data_Type': str(col_data.dtype),
            'Non_Null_Count': non_null_count,
            'Missing_Count': missing_count,
            'Missing_Percentage': missing_percentage,
            'Unique_Values': unique_values,
            'Min_Value': min_val,
            'Max_Value': max_val,
            'Mean': mean_val,
            'Std': std_val,
            'Sample_Value': sample_val
        })
    
    # Create the features dataframe
    features_df = pd.DataFrame(features_summary)
    
    # Display summary statistics
    print(f"📊 FEATURES OVERVIEW:")
    print(f"   Total Features Extracted: {len(features_df)}")
    print(f"   Features by Category:")
    
    category_counts = features_df['Category'].value_counts()
    category_summary = pd.DataFrame({
        'Category': category_counts.index,
        'Count': category_counts.values,
        'Percentage': [safe_round((count / category_counts.sum()) * 100, 1) for count in category_counts.values]
    })
    
    from IPython.display import display
    display(category_summary)
    
    print(f"\n📋 DETAILED FEATURES TABLE:")
    print("-" * 60)
    
    # Display the features dataframe with basic styling
    display(features_df)
    
    # Sample data with key features
    print(f"\n📋 SAMPLE DATA WITH KEY FEATURES:")
    print("-" * 60)
    
    # Select key features for sample display
    key_features = []
    for category in ['Prompt Features', 'Response A Features', 'Response B Features']:
        cat_features = features_df[features_df['Category'] == category]['Feature_Name'].tolist()
        # Take first few features from each category
        key_features.extend(cat_features[:3])
    
    # Add some specific important features
    important_features = [col for col in feature_cols if any(x in col for x in 
                          ['char_count', 'word_count', 'has_code', 'has_markdown'])]
    key_features.extend(important_features[:5])
    
    # Remove duplicates and filter existing columns
    key_features = list(dict.fromkeys(key_features))  # Remove duplicates
    existing_key_features = [col for col in key_features if col in train_df.columns]
    
    # Create sample dataframe
    if existing_key_features:
        sample_features_df = train_df[['id'] + existing_key_features[:15]].head(10)
        display(sample_features_df)
    
        # Export features metadata to CSV
        features_df.to_csv('extracted_features_metadata.csv', index=False)
        sample_features_df.to_csv('sample_features_data.csv', index=False)
        
        print(f"\n✅ Features analysis completed!")
        print(f"📁 Exported files:")
        print(f"   - extracted_features_metadata.csv ({len(features_df)} features)")
        print(f"   - sample_features_data.csv (sample data with {len(existing_key_features[:15])} key features)")
    else:
        print("⚠️  No key features found for sample display")

# 3. 🎯 Target Distribution & Preference Analysis

In [ ]:
if has_targets:
    print("🎯 COMPREHENSIVE TARGET ANALYSIS")
    print("=" * 45)
    
    # Basic target statistics
    print("\n📊 Target Distribution Statistics:")
    target_stats = train_df[target_cols].describe()
    print(target_stats)
    
    # Preference patterns
    train_df['preferred_class'] = train_df[target_cols].idxmax(axis=1)
    train_df['preference_confidence'] = train_df[target_cols].max(axis=1)
    train_df['preference_margin'] = train_df[target_cols].apply(lambda x: x.nlargest(2).iloc[0] - x.nlargest(2).iloc[1], axis=1)
    
    print("\n🏆 Overall Preference Distribution:")
    pref_dist = train_df['preferred_class'].value_counts()
    for pref, count in pref_dist.items():
        print(f"  {pref}: {count:,} ({count/len(train_df)*100:.1f}%)")
    
    print(f"\n🎖️  Preference Confidence:")
    conf_stats = train_df['preference_confidence'].describe()
    print(f"  Mean: {conf_stats['mean']:.4f}")
    print(f"  Std:  {conf_stats['std']:.4f}")
    print(f"  Min:  {conf_stats['min']:.4f}")
    print(f"  Max:  {conf_stats['max']:.4f}")
    
    print(f"\n📏 Preference Margin (difference between 1st and 2nd choice):")
    margin_stats = train_df['preference_margin'].describe()
    print(f"  Mean: {margin_stats['mean']:.4f}")
    print(f"  Std:  {margin_stats['std']:.4f}")
    
    # High vs Low confidence examples
    high_conf_threshold = train_df['preference_confidence'].quantile(0.8)
    low_conf_threshold = train_df['preference_confidence'].quantile(0.2)
    
    high_conf_count = (train_df['preference_confidence'] >= high_conf_threshold).sum()
    low_conf_count = (train_df['preference_confidence'] <= low_conf_threshold).sum()
    
    print(f"\n🔥 High Confidence (≥{high_conf_threshold:.3f}): {high_conf_count:,} examples ({high_conf_count/len(train_df)*100:.1f}%)")
    print(f"❓ Low Confidence (≤{low_conf_threshold:.3f}): {low_conf_count:,} examples ({low_conf_count/len(train_df)*100:.1f}%)")
    
    # Analyze confidence by preference type
    print(f"\n📈 Confidence by Preference Type:")
    for pref_type in ['winner_model_a', 'winner_model_b', 'winner_tie']:
        mask = train_df['preferred_class'] == pref_type
        if mask.sum() > 0:
            mean_conf = train_df.loc[mask, 'preference_confidence'].mean()
            print(f"  {pref_type}: {mean_conf:.4f} average confidence")
    
else:
    print("❌ No target columns found - cannot perform preference analysis")
    print("ℹ️  This appears to be a test-only dataset or the labels are in a different format")

# 4. 📏 Text Length & Complexity Analysis

In [ ]:
# Comprehensive text length analysis
print("📏 TEXT LENGTH & COMPLEXITY ANALYSIS")
print("=" * 40)

# Calculate comparative features
if all(col in train_df.columns for col in ['response_a_text_char_count', 'response_b_text_char_count']):
    train_df['response_length_diff'] = train_df['response_a_text_char_count'] - train_df['response_b_text_char_count']
    train_df['response_length_ratio'] = train_df['response_a_text_char_count'] / (train_df['response_b_text_char_count'] + 1)
    train_df['response_word_diff'] = train_df['response_a_text_word_count'] - train_df['response_b_text_word_count']
    train_df['response_word_ratio'] = train_df['response_a_text_word_count'] / (train_df['response_b_text_word_count'] + 1)

# Length statistics
length_features = [col for col in train_df.columns if any(x in col for x in ['char_count', 'word_count', 'sentence_count'])]

if length_features:
    print("\n📊 Length Feature Statistics:")
    length_stats = train_df[length_features].describe()
    print(length_stats.round(2))
    
    print("\n🔍 Comparative Analysis:")
    if 'response_length_diff' in train_df.columns:
        print(f"Response Length Difference (A - B):")
        print(f"  Mean: {train_df['response_length_diff'].mean():.1f} characters")
        print(f"  Std:  {train_df['response_length_diff'].std():.1f}")
        
        # Count preferences
        longer_a = (train_df['response_length_diff'] > 0).sum()
        equal_length = (train_df['response_length_diff'] == 0).sum()
        longer_b = (train_df['response_length_diff'] < 0).sum()
        
        print(f"\n📊 Length Comparison:")
        print(f"  Response A longer: {longer_a:,} ({longer_a/len(train_df)*100:.1f}%)")
        print(f"  Equal length: {equal_length:,} ({equal_length/len(train_df)*100:.1f}%)")
        print(f"  Response B longer: {longer_b:,} ({longer_b/len(train_df)*100:.1f}%)")
    
    if 'response_length_ratio' in train_df.columns:
        print(f"\nResponse Length Ratio (A/B):")
        print(f"  Mean: {train_df['response_length_ratio'].mean():.2f}")
        print(f"  Median: {train_df['response_length_ratio'].median():.2f}")
        
        # Extreme ratios
        very_long_a = (train_df['response_length_ratio'] > 2).sum()
        very_long_b = (train_df['response_length_ratio'] < 0.5).sum()
        print(f"  A much longer (>2x): {very_long_a:,} ({very_long_a/len(train_df)*100:.1f}%)")
        print(f"  B much longer (<0.5x): {very_long_b:,} ({very_long_b/len(train_df)*100:.1f}%)")

else:
    print("⚠️  Length features not found - may need to run text processing first")


In [ ]:
# Comprehensive length visualizations
if length_features:
    fig = plt.figure(figsize=(20, 12))
    
    # Character count distributions
    ax1 = plt.subplot(3, 4, 1)
    if 'prompt_text_char_count' in train_df.columns:
        ax1.hist(train_df['prompt_text_char_count'], bins=50, alpha=0.7, edgecolor='black')
        ax1.set_title('Prompt Character Count', fontweight='bold')
        ax1.set_xlabel('Characters')
        ax1.set_yscale('log')
    
    ax2 = plt.subplot(3, 4, 2)
    if all(col in train_df.columns for col in ['response_a_text_char_count', 'response_b_text_char_count']):
        ax2.hist(train_df['response_a_text_char_count'], bins=50, alpha=0.7, label='Response A', edgecolor='black')
        ax2.hist(train_df['response_b_text_char_count'], bins=50, alpha=0.7, label='Response B', edgecolor='black')
        ax2.set_title('Response Character Counts', fontweight='bold')
        ax2.set_xlabel('Characters')
        ax2.set_yscale('log')
        ax2.legend()
    
    # Word count distributions
    ax3 = plt.subplot(3, 4, 3)
    if 'prompt_text_word_count' in train_df.columns:
        ax3.hist(train_df['prompt_text_word_count'], bins=50, alpha=0.7, edgecolor='black')
        ax3.set_title('Prompt Word Count', fontweight='bold')
        ax3.set_xlabel('Words')
        ax3.set_yscale('log')
    
    ax4 = plt.subplot(3, 4, 4)
    if all(col in train_df.columns for col in ['response_a_text_word_count', 'response_b_text_word_count']):
        ax4.hist(train_df['response_a_text_word_count'], bins=50, alpha=0.7, label='Response A', edgecolor='black')
        ax4.hist(train_df['response_b_text_word_count'], bins=50, alpha=0.7, label='Response B', edgecolor='black')
        ax4.set_title('Response Word Counts', fontweight='bold')
        ax4.set_xlabel('Words')
        ax4.set_yscale('log')
        ax4.legend()
    
    # Length differences
    ax5 = plt.subplot(3, 4, 5)
    if 'response_length_diff' in train_df.columns:
        ax5.hist(train_df['response_length_diff'], bins=50, alpha=0.7, edgecolor='black')
        ax5.axvline(0, color='red', linestyle='--', label='Equal length')
        ax5.axvline(train_df['response_length_diff'].mean(), color='orange', linestyle='--', 
                   label=f'Mean: {train_df["response_length_diff"].mean():.0f}')
        ax5.set_title('Character Length Difference (A - B)', fontweight='bold')
        ax5.set_xlabel('Character Difference')
        ax5.legend()
    
    # Length ratios
    ax6 = plt.subplot(3, 4, 6)
    if 'response_length_ratio' in train_df.columns:
        # Clip extreme values for better visualization
        ratio_clipped = np.clip(train_df['response_length_ratio'], 0, 5)
        ax6.hist(ratio_clipped, bins=50, alpha=0.7, edgecolor='black')
        ax6.axvline(1, color='red', linestyle='--', label='Equal length')
        ax6.axvline(train_df['response_length_ratio'].median(), color='orange', linestyle='--',
                   label=f'Median: {train_df["response_length_ratio"].median():.2f}')
        ax6.set_title('Length Ratio (A/B, clipped at 5)', fontweight='bold')
        ax6.set_xlabel('Ratio')
        ax6.legend()
    
    # Scatter plot: Response A vs B lengths
    ax7 = plt.subplot(3, 4, 7)
    if all(col in train_df.columns for col in ['response_a_text_char_count', 'response_b_text_char_count']):
        sample_size = min(5000, len(train_df))  # Sample for performance
        sample_idx = np.random.choice(len(train_df), sample_size, replace=False)
        sample_data = train_df.iloc[sample_idx]
        
        ax7.scatter(sample_data['response_a_text_char_count'], sample_data['response_b_text_char_count'], 
                   alpha=0.6, s=2)
        
        # Add diagonal line
        max_len = max(sample_data['response_a_text_char_count'].max(), sample_data['response_b_text_char_count'].max())
        ax7.plot([0, max_len], [0, max_len], 'r--', alpha=0.7, label='Equal length')
        
        ax7.set_xlabel('Response A Character Count')
        ax7.set_ylabel('Response B Character Count')
        ax7.set_title('Response A vs B Lengths', fontweight='bold')
        ax7.legend()
    
    # Sentence complexity comparison
    ax8 = plt.subplot(3, 4, 8)
    if all(col in train_df.columns for col in ['response_a_text_sentence_count', 'response_b_text_sentence_count']):
        ax8.hist(train_df['response_a_text_sentence_count'], bins=30, alpha=0.7, label='Response A', edgecolor='black')
        ax8.hist(train_df['response_b_text_sentence_count'], bins=30, alpha=0.7, label='Response B', edgecolor='black')
        ax8.set_title('Sentence Count Comparison', fontweight='bold')
        ax8.set_xlabel('Sentence Count')
        ax8.set_yscale('log')
        ax8.legend()
    
    # If we have targets, show length preferences
    if has_targets and 'response_length_diff' in train_df.columns:
        ax9 = plt.subplot(3, 4, 9)
        for i, pref in enumerate(['winner_model_a', 'winner_model_b', 'winner_tie']):
            mask = train_df['preferred_class'] == pref
            if mask.sum() > 0:
                ax9.hist(train_df.loc[mask, 'response_length_diff'], bins=30, alpha=0.7, 
                        label=f'{pref} ({mask.sum()})', density=True)
        ax9.axvline(0, color='black', linestyle='--', alpha=0.5)
        ax9.set_title('Length Difference by Preference', fontweight='bold')
        ax9.set_xlabel('Length Difference (A - B)')
        ax9.legend()
        
        # Average word length comparison
        ax10 = plt.subplot(3, 4, 10)
        if all(col in train_df.columns for col in ['response_a_text_avg_word_length', 'response_b_text_avg_word_length']):
            ax10.hist(train_df['response_a_text_avg_word_length'], bins=30, alpha=0.7, 
                     label='Response A', edgecolor='black')
            ax10.hist(train_df['response_b_text_avg_word_length'], bins=30, alpha=0.7, 
                     label='Response B', edgecolor='black')
            ax10.set_title('Average Word Length', fontweight='bold')
            ax10.set_xlabel('Avg Word Length')
            ax10.legend()
        
        # Length vs confidence correlation
        ax11 = plt.subplot(3, 4, 11)
        if 'preference_confidence' in train_df.columns:
            sample_size = min(3000, len(train_df))
            sample_idx = np.random.choice(len(train_df), sample_size, replace=False)
            sample_data = train_df.iloc[sample_idx]
            
            ax11.scatter(sample_data['response_length_diff'], sample_data['preference_confidence'], 
                        alpha=0.6, s=2)
            ax11.axvline(0, color='red', linestyle='--', alpha=0.5)
            ax11.set_xlabel('Length Difference (A - B)')
            ax11.set_ylabel('Preference Confidence')
            ax11.set_title('Length vs Confidence', fontweight='bold')
            
        # Punctuation usage comparison
        ax12 = plt.subplot(3, 4, 12)
        if all(col in train_df.columns for col in ['response_a_text_punct_count', 'response_b_text_punct_count']):
            train_df['punct_ratio_a'] = train_df['response_a_text_punct_count'] / (train_df['response_a_text_char_count'] + 1)
            train_df['punct_ratio_b'] = train_df['response_b_text_punct_count'] / (train_df['response_b_text_char_count'] + 1)
            
            ax12.hist(train_df['punct_ratio_a'], bins=30, alpha=0.7, label='Response A', edgecolor='black')
            ax12.hist(train_df['punct_ratio_b'], bins=30, alpha=0.7, label='Response B', edgecolor='black')
            ax12.set_title('Punctuation Ratio', fontweight='bold')
            ax12.set_xlabel('Punctuation / Characters')
            ax12.legend()
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Length analysis visualization completed!")
    
else:
    print("⚠️  Skipping length visualization - features not available")

# 5. 🔍 Content Analysis & Quality Indicators

In [ ]:
# Content type and quality analysis
print("🔍 CONTENT ANALYSIS & QUALITY INDICATORS")
print("=" * 45)

# Analyze content types
content_features = [col for col in train_df.columns if any(x in col for x in ['has_code', 'has_markdown', 'has_links', 'has_lists', 'has_tables', 'has_math'])]

if content_features:
    print("\n📊 Content Type Distribution:")
    
    # Response A content analysis
    print("\nResponse A content types:")
    for feature in ['response_a_text_has_code', 'response_a_text_has_markdown', 'response_a_text_has_links', 
                   'response_a_text_has_lists', 'response_a_text_has_tables', 'response_a_text_has_math']:
        if feature in train_df.columns:
            count = train_df[feature].sum()
            pct = count / len(train_df) * 100
            feature_name = feature.replace('response_a_text_has_', '').title()
            print(f"  {feature_name}: {count:,} ({pct:.1f}%)")
    
    # Response B content analysis
    print("\nResponse B content types:")
    for feature in ['response_b_text_has_code', 'response_b_text_has_markdown', 'response_b_text_has_links', 
                   'response_b_text_has_lists', 'response_b_text_has_tables', 'response_b_text_has_math']:
        if feature in train_df.columns:
            count = train_df[feature].sum()
            pct = count / len(train_df) * 100
            feature_name = feature.replace('response_b_text_has_', '').title()
            print(f"  {feature_name}: {count:,} ({pct:.1f}%)")
    
    # Comparative content analysis
    print("\n🔄 Comparative Content Analysis:")
    
    content_comparisons = [
        ('has_code', 'Code blocks/snippets'),
        ('has_markdown', 'Markdown formatting'),
        ('has_lists', 'Lists/enumeration'),
        ('has_links', 'External links'),
        ('has_tables', 'Tables/structured data'),
        ('has_math', 'Mathematical expressions')
    ]
    
    for content_type, description in content_comparisons:
        col_a = f'response_a_text_{content_type}'
        col_b = f'response_b_text_{content_type}'
        
        if col_a in train_df.columns and col_b in train_df.columns:
            both_have = (train_df[col_a] & train_df[col_b]).sum()
            only_a = (train_df[col_a] & ~train_df[col_b]).sum()
            only_b = (~train_df[col_a] & train_df[col_b]).sum()
            neither = (~train_df[col_a] & ~train_df[col_b]).sum()
            
            print(f"\n{description}:")
            print(f"  Both have: {both_have:,} ({both_have/len(train_df)*100:.1f}%)")
            print(f"  Only A has: {only_a:,} ({only_a/len(train_df)*100:.1f}%)")
            print(f"  Only B has: {only_b:,} ({only_b/len(train_df)*100:.1f}%)")
            print(f"  Neither has: {neither:,} ({neither/len(train_df)*100:.1f}%)")
            
            # Calculate preference based on content
            if has_targets and only_a > 0 and only_b > 0:
                a_wins_when_only_a = train_df[train_df[col_a] & ~train_df[col_b]]['preferred_class'].value_counts()
                b_wins_when_only_b = train_df[~train_df[col_a] & train_df[col_b]]['preferred_class'].value_counts()
                
                if 'winner_model_a' in a_wins_when_only_a.index:
                    a_advantage = a_wins_when_only_a.get('winner_model_a', 0) / only_a * 100
                    print(f"  A wins when only A has {content_type}: {a_advantage:.1f}%")
                
                if 'winner_model_b' in b_wins_when_only_b.index:
                    b_advantage = b_wins_when_only_b.get('winner_model_b', 0) / only_b * 100
                    print(f"  B wins when only B has {content_type}: {b_advantage:.1f}%")

# Readability analysis
readability_features = [col for col in train_df.columns if any(x in col for x in ['flesch_ease', 'flesch_kincaid', 'automated_readability'])]

if readability_features and TEXTSTAT_AVAILABLE:
    print(f"\n📖 READABILITY ANALYSIS")
    print("-" * 25)
    
    # Filter out NaN values for analysis
    for score_type in ['flesch_ease', 'flesch_kincaid', 'automated_readability']:
        col_a = f'response_a_text_{score_type}'
        col_b = f'response_b_text_{score_type}'
        
        if col_a in train_df.columns and col_b in train_df.columns:
            # Remove NaN values
            valid_a = train_df[col_a].dropna()
            valid_b = train_df[col_b].dropna()
            
            if len(valid_a) > 0 and len(valid_b) > 0:
                print(f"\n{score_type.replace('_', ' ').title()}:")
                print(f"  Response A - Mean: {valid_a.mean():.2f}, Std: {valid_a.std():.2f}")
                print(f"  Response B - Mean: {valid_b.mean():.2f}, Std: {valid_b.std():.2f}")
                
                # Statistical test
                from scipy import stats
                stat, p_value = stats.ttest_ind(valid_a, valid_b)
                significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
                print(f"  Difference significance: {significance} (p={p_value:.4f})")
                
                # Calculate readability difference
                common_indices = train_df[col_a].notna() & train_df[col_b].notna()
                if common_indices.sum() > 0:
                    readability_diff = train_df.loc[common_indices, col_a] - train_df.loc[common_indices, col_b]
                    print(f"  Average difference (A-B): {readability_diff.mean():.2f}")

else:
    if not TEXTSTAT_AVAILABLE:
        print("\n📖 Readability analysis skipped - textstat not available")
        print("   Install with: pip install textstat")
    else:
        print("\n📖 Readability features not found - may need to run text processing first")


# 6. 📋 Sample Analysis & Pattern Discovery

In [ ]:
# Comprehensive sample analysis
print("📋 SAMPLE ANALYSIS & PATTERN DISCOVERY")
print("=" * 42)

def display_example(idx, title, show_features=True):
    """Display a comprehensive example with analysis"""
    print(f"\n{title}")
    print("=" * len(title))
    
    row = train_df.iloc[idx]
    print(f"🆔 ID: {row['id']}")
    
    # Display texts
    if 'prompt_text' in train_df.columns:
        prompt = str(row['prompt_text'])
        print(f"\n📝 PROMPT ({len(prompt)} chars):")
        print(f"   {prompt[:300]}{'...' if len(prompt) > 300 else ''}")
    
    if 'response_a_text' in train_df.columns:
        response_a = str(row['response_a_text'])
        print(f"\n🤖 RESPONSE A ({len(response_a)} chars):")
        print(f"   {response_a[:300]}{'...' if len(response_a) > 300 else ''}")
    
    if 'response_b_text' in train_df.columns:
        response_b = str(row['response_b_text'])
        print(f"\n🤖 RESPONSE B ({len(response_b)} chars):")
        print(f"   {response_b[:300]}{'...' if len(response_b) > 300 else ''}")
    
    # Show target information
    if has_targets:
        print(f"\n🎯 TARGETS:")
        for col in target_cols:
            print(f"   {col}: {row[col]:.4f}")
        
        if 'preferred_class' in train_df.columns:
            print(f"   🏆 Winner: {row['preferred_class']}")
            print(f"   🎖️  Confidence: {row['preference_confidence']:.4f}")
            print(f"   📏 Margin: {row['preference_margin']:.4f}")
    
    # Show key features if available
    if show_features:
        print(f"\n📊 KEY FEATURES:")
        
        # Length features
        if 'response_length_diff' in train_df.columns:
            print(f"   Length difference (A-B): {row['response_length_diff']:.0f} chars")
        if 'response_length_ratio' in train_df.columns:
            print(f"   Length ratio (A/B): {row['response_length_ratio']:.2f}")
        
        # Content features
        content_types = ['has_code', 'has_markdown', 'has_lists', 'has_links']
        for content in content_types:
            col_a = f'response_a_text_{content}'
            col_b = f'response_b_text_{content}'
            if col_a in train_df.columns and col_b in train_df.columns:
                a_has = row[col_a]
                b_has = row[col_b]
                if a_has or b_has:
                    status = f"A: {'✓' if a_has else '✗'}, B: {'✓' if b_has else '✗'}"
                    print(f"   {content.replace('has_', '').title()}: {status}")

# Show diverse examples
if len(train_df) > 0:
    # Random example
    display_example(0, "🎲 EXAMPLE 1: First Entry")
    
    if len(train_df) > 1:
        display_example(1, "🎲 EXAMPLE 2: Second Entry")
    
    if has_targets and len(train_df) > 10:
        # High confidence example
        if 'preference_confidence' in train_df.columns:
            high_conf_idx = train_df['preference_confidence'].idxmax()
            display_example(high_conf_idx, "🔥 EXAMPLE 3: Highest Confidence Prediction")
            
            # Low confidence example
            low_conf_idx = train_df['preference_confidence'].idxmin()
            display_example(low_conf_idx, "❓ EXAMPLE 4: Lowest Confidence Prediction")
        
        # Large length difference example
        if 'response_length_diff' in train_df.columns:
            large_diff_idx = train_df['response_length_diff'].abs().idxmax()
            display_example(large_diff_idx, "📏 EXAMPLE 5: Largest Length Difference")
        
        # Code example (if available)
        if 'response_a_text_has_code' in train_df.columns:
            code_mask = train_df['response_a_text_has_code'] | train_df['response_b_text_has_code']
            if code_mask.sum() > 0:
                code_idx = train_df[code_mask].index[0]
                display_example(code_idx, "💻 EXAMPLE 6: Contains Code")

print(f"\n✅ Sample analysis completed!")

# Pattern discovery
if has_targets and len(train_df) > 100:
    print(f"\n🔍 PATTERN DISCOVERY")
    print("-" * 20)
    
    # Length bias analysis
    if 'response_length_diff' in train_df.columns:
        print(f"\n📏 Length Bias Analysis:")
        
        # Correlation between length difference and preference
        length_pref_corr = train_df['response_length_diff'].corr(train_df['winner_model_a'] - train_df['winner_model_b'])
        print(f"   Length difference vs A preference: r = {length_pref_corr:.4f}")
        
        # Mean length difference by preference
        for pref in ['winner_model_a', 'winner_model_b', 'winner_tie']:
            mask = train_df['preferred_class'] == pref
            if mask.sum() > 0:
                mean_diff = train_df.loc[mask, 'response_length_diff'].mean()
                print(f"   {pref}: avg length diff = {mean_diff:.1f}")
    
    # Content advantage analysis
    print(f"\n🎨 Content Advantage Analysis:")
    for content_type in ['has_code', 'has_markdown', 'has_lists']:
        col_a = f'response_a_text_{content_type}'
        col_b = f'response_b_text_{content_type}'
        
        if col_a in train_df.columns and col_b in train_df.columns:
            # When only A has this content type
            only_a_mask = train_df[col_a] & ~train_df[col_b]
            if only_a_mask.sum() > 0:
                a_win_rate = (train_df.loc[only_a_mask, 'preferred_class'] == 'winner_model_a').mean()
                print(f"   When only A has {content_type}: A wins {a_win_rate*100:.1f}% of time")
            
            # When only B has this content type
            only_b_mask = ~train_df[col_a] & train_df[col_b]
            if only_b_mask.sum() > 0:
                b_win_rate = (train_df.loc[only_b_mask, 'preferred_class'] == 'winner_model_b').mean()
                print(f"   When only B has {content_type}: B wins {b_win_rate*100:.1f}% of time")
    
    # Confidence patterns
    if 'preference_confidence' in train_df.columns:
        print(f"\n🎖️  Confidence Patterns:")
        conf_quartiles = train_df['preference_confidence'].quantile([0.25, 0.5, 0.75])
        
        for q, threshold in zip(['Low', 'Medium', 'High'], [conf_quartiles[0.25], conf_quartiles[0.5], conf_quartiles[0.75]]):
            if q == 'Low':
                mask = train_df['preference_confidence'] <= threshold
            elif q == 'Medium':
                mask = (train_df['preference_confidence'] > conf_quartiles[0.25]) & (train_df['preference_confidence'] <= threshold)
            else:
                mask = train_df['preference_confidence'] > threshold
            
            if mask.sum() > 0:
                pref_dist = train_df.loc[mask, 'preferred_class'].value_counts(normalize=True)
                print(f"   {q} confidence group ({mask.sum()} examples):")
                for pref, pct in pref_dist.items():
                    print(f"     {pref}: {pct*100:.1f}%")


# 7. 🎯 Modeling Strategy & Recommendations

In [ ]:
# Generate modeling recommendations based on EDA findings
print("🎯 MODELING STRATEGY & RECOMMENDATIONS")
print("=" * 42)

# Create comprehensive feature summary
modeling_features = []

# Basic comparative features
comparative_features = [
    'response_length_diff', 'response_length_ratio', 'response_word_diff', 'response_word_ratio'
]

for feature in comparative_features:
    if feature in train_df.columns:
        modeling_features.append(feature)

# Content comparison features  
content_types = ['has_code', 'has_markdown', 'has_lists', 'has_links', 'has_tables', 'has_math']
for content in content_types:
    col_a = f'response_a_text_{content}'
    col_b = f'response_b_text_{content}'
    if col_a in train_df.columns and col_b in train_df.columns:
        # Create comparison features
        train_df[f'{content}_advantage_a'] = train_df[col_a] & ~train_df[col_b]
        train_df[f'{content}_advantage_b'] = ~train_df[col_a] & train_df[col_b]
        train_df[f'{content}_both'] = train_df[col_a] & train_df[col_b]
        
        modeling_features.extend([f'{content}_advantage_a', f'{content}_advantage_b', f'{content}_both'])

# Quality comparison features
quality_features = ['avg_word_length', 'punct_count', 'sentence_count', 'paragraph_count']
for quality in quality_features:
    col_a = f'response_a_text_{quality}'
    col_b = f'response_b_text_{quality}'
    if col_a in train_df.columns and col_b in train_df.columns:
        train_df[f'{quality}_diff'] = train_df[col_a] - train_df[col_b]
        train_df[f'{quality}_ratio'] = train_df[col_a] / (train_df[col_b] + 1)
        modeling_features.extend([f'{quality}_diff', f'{quality}_ratio'])

# Readability comparison
if TEXTSTAT_AVAILABLE:
    readability_types = ['flesch_ease', 'flesch_kincaid', 'automated_readability']
    for readability in readability_types:
        col_a = f'response_a_text_{readability}'
        col_b = f'response_b_text_{readability}'
        if col_a in train_df.columns and col_b in train_df.columns:
            # Only for non-NaN pairs
            valid_mask = train_df[col_a].notna() & train_df[col_b].notna()
            if valid_mask.sum() > 0:
                train_df[f'{readability}_diff'] = np.nan
                train_df.loc[valid_mask, f'{readability}_diff'] = train_df.loc[valid_mask, col_a] - train_df.loc[valid_mask, col_b]
                modeling_features.append(f'{readability}_diff')

# Filter to existing features
existing_features = [f for f in modeling_features if f in train_df.columns]

print(f"\n📊 FEATURE ENGINEERING SUMMARY")
print("-" * 30)
print(f"Total engineered features: {len(existing_features)}")
print(f"Feature categories:")
print(f"  - Length comparisons: {len([f for f in existing_features if 'length' in f or 'word' in f])}")
print(f"  - Content advantages: {len([f for f in existing_features if 'advantage' in f or 'both' in f])}")
print(f"  - Quality differences: {len([f for f in existing_features if 'diff' in f and 'length' not in f])}")
print(f"  - Quality ratios: {len([f for f in existing_features if 'ratio' in f and 'length' not in f])}")

print(f"\n🎯 MODELING APPROACH RECOMMENDATIONS")
print("-" * 35)

print(f"\n1️⃣ BASELINE APPROACH: Feature Engineering + Traditional ML")
print(f"   ✅ Use engineered comparative features")
print(f"   ✅ XGBoost/LightGBM for 3-class classification")
print(f"   ✅ Fast iteration and interpretable results")
print(f"   📊 Expected performance: Moderate but reliable")

print(f"\n2️⃣ INTERMEDIATE APPROACH: Transformer Fine-tuning")
print(f"   ✅ Concatenate prompt + response_a + response_b")
print(f"   ✅ Fine-tune BERT/RoBERTa/DeBERTa for classification")
print(f"   ✅ Add engineered features as additional inputs")
print(f"   📊 Expected performance: Good semantic understanding")

print(f"\n3️⃣ ADVANCED APPROACH: Specialized Architecture")
print(f"   ✅ Dual-encoder with cross-attention")
print(f"   ✅ Ranking-based loss functions")
print(f"   ✅ Knowledge distillation from larger models")
print(f"   📊 Expected performance: Best but resource-intensive")

# Data insights for modeling
if has_targets:
    print(f"\n📈 KEY INSIGHTS FOR MODELING")
    print("-" * 27)
    
    # Class balance insights
    class_dist = train_df['preferred_class'].value_counts(normalize=True)
    most_common = class_dist.index[0]
    most_common_pct = class_dist.iloc[0]
    
    print(f"\n🎯 Target Distribution:")
    print(f"   Most common class: {most_common} ({most_common_pct*100:.1f}%)")
    print(f"   Class balance: {'Balanced' if most_common_pct < 0.4 else 'Imbalanced'}")
    
    if most_common_pct > 0.5:
        print(f"   ⚠️  Consider class weighting or resampling")
    
    # Confidence insights
    if 'preference_confidence' in train_df.columns:
        low_conf_pct = (train_df['preference_confidence'] < 0.4).mean()
        high_conf_pct = (train_df['preference_confidence'] > 0.7).mean()
        
        print(f"\n🎖️  Confidence Patterns:")
        print(f"   Low confidence (<0.4): {low_conf_pct*100:.1f}%")
        print(f"   High confidence (>0.7): {high_conf_pct*100:.1f}%")
        
        if low_conf_pct > 0.3:
            print(f"   💡 Many ambiguous cases - focus on uncertainty estimation")
        
        if high_conf_pct > 0.3:
            print(f"   💡 Many clear cases - good for knowledge distillation")
    
    # Feature importance hints
    if 'response_length_diff' in train_df.columns:
        length_bias = abs(train_df['response_length_diff'].corr(train_df['winner_model_a'] - train_df['winner_model_b']))
        if length_bias > 0.1:
            print(f"\n📏 Length bias detected (r={length_bias:.3f})")
            print(f"   💡 Length features will be important")
    
    # Content type insights
    content_advantages = []
    for content in ['has_code', 'has_markdown', 'has_lists']:
        col_a = f'response_a_text_{content}'
        col_b = f'response_b_text_{content}'
        if col_a in train_df.columns and col_b in train_df.columns:
            # Calculate advantage when only one has the content
            only_a_mask = train_df[col_a] & ~train_df[col_b]
            only_b_mask = ~train_df[col_a] & train_df[col_b]
            
            if only_a_mask.sum() > 0:
                a_advantage = (train_df.loc[only_a_mask, 'preferred_class'] == 'winner_model_a').mean()
                if a_advantage > 0.6:  # Strong advantage
                    content_advantages.append(f"{content} (A advantage: {a_advantage*100:.0f}%)")
            
            if only_b_mask.sum() > 0:
                b_advantage = (train_df.loc[only_b_mask, 'preferred_class'] == 'winner_model_b').mean()
                if b_advantage > 0.6:  # Strong advantage
                    content_advantages.append(f"{content} (B advantage: {b_advantage*100:.0f}%)")
    
    if content_advantages:
        print(f"\n🎨 Strong Content Advantages:")
        for advantage in content_advantages:
            print(f"   💡 {advantage}")

print(f"\n🔧 TECHNICAL IMPLEMENTATION TIPS")
print("-" * 32)
print(f"📊 Validation Strategy:")
print(f"   ✅ Stratified K-fold based on preferred_class")
print(f"   ✅ Consider GroupKFold if prompt similarity exists")
print(f"   ✅ Monitor log-loss for probability predictions")

print(f"\n🧠 Model Selection:")
print(f"   ✅ Start with XGBoost baseline using engineered features")
print(f"   ✅ Try RoBERTa-base or DeBERTa-v3-base for transformers")
print(f"   ✅ Ensemble different model types for final submission")

print(f"\n⚡ Advanced Techniques (from winning solutions):")
print(f"   🎯 Knowledge Distillation: Train large → distill to small")
print(f"   🎯 Pseudo-labeling: Use external human preference datasets")
print(f"   🎯 Post-pretraining: Domain-adapt before fine-tuning")
print(f"   🎯 Test-Time Augmentation: Multiple inference passes")

print(f"\n🚀 Next Steps:")
print(f"   1. Implement XGBoost baseline with current features")
print(f"   2. Fine-tune transformer models on text data")
print(f"   3. Combine predictions using ensemble methods")
print(f"   4. Apply advanced techniques for final boost")

print(f"\n✅ Modeling recommendations completed!")
